# Solving a complex task with a multi-agent hierarchy

In [3]:
!pip install 'smolagents[litellm]' plotly geopandas shapely kaleido -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 14.7 MB/s eta 0:00:00


In [ ]:

from huggingface_hub import login

login("")


In [4]:
# We first make a tool to get the cargo plane transfer time.
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth
    using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h
            (defaults to 750 km/h for typical cargo planes)


    """
    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)


    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)


    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0


    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)

print(calculate_cargo_travel_time((41.8781, -87.6298),(-33.8688, 151.2093),))

22.82


For the model provider, we use Together AI, one of the new inference providers on the Hub!

Regarding the GoogleSearchTool: this requires either having the environment
variable SERPAPI_API_KEY and passing provider="serpapi"
or having SERPER_API_KEY and passing provider="serper".

If you don't have any SERP API provider setup, you can use
DuckDuckGoSearchTool, but be aware that it has a rate limit.

In [5]:
!pip install smolagents[telemetry] \
    opentelemetry-sdk \
    opentelemetry-exporter-otlp \
    openinference-instrumentation-smolagents


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 1.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-instrumentation to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.5/330.5 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.4/121.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import base64
import os

os.environ["LANGFUSE_PUBLIC_KEY"] = "your_public_key_here"
os.environ["LANGFUSE_SECRET_KEY"] = "your_secret_key"

LANGFUSE_PUBLIC_KEY = os.environ.get("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.environ.get("LANGFUSE_SECRET_KEY")

if not LANGFUSE_PUBLIC_KEY or not LANGFUSE_SECRET_KEY:
    raise ValueError("Langfuse public or secret keys are missing!")

LANGFUSE_AUTH = base64.b64encode(
    f"{LANGFUSE_PUBLIC_KEY}:{LANGFUSE_SECRET_KEY}".encode()
).decode()

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://us.cloud.langfuse.com/api/public/otel"  # US data region

os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = (
    f"Authorization=Basic {LANGFUSE_AUTH},x-langfuse-ingestion-version=4"
)

print("Langfuse environment variables set successfully!")

Langfuse environment variables set successfully!


In [7]:
!pip install -U smolagents "smolagents[openai]" openai -q
from opentelemetry.sdk.trace import TracerProvider

from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

trace_provider = TracerProvider()
trace_provider.add_span_processor(
    SimpleSpanProcessor(OTLPSpanExporter())
)

SmolagentsInstrumentor().instrument(
    tracer_provider=trace_provider
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.1 MB/s eta 0:00:00


In [8]:
import os
from PIL import Image
from smolagents import CodeAgent, GoogleSearchTool, InferenceClientModel, VisitWebpageTool


model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")

We can start with creating a baseline, simple agent to give us a simple report.

In [8]:
task = """Find all Harry Potter filming locations in the world, calculate the time to transfer via passenger plane to here
Also give me some cricket stadiums if there are any nearby to the filming locations"""

In [ ]:
from google.colab import userdata
import os

os.environ["SERPAPI_API_KEY"] = "your_serpapi_api_key_here"

In [10]:
agent = CodeAgent(
    model=model,
    tools=[GoogleSearchTool(), VisitWebpageTool(), calculate_cargo_travel_time],
    additional_authorized_imports=["pandas"],
    max_steps=5,
)

In [11]:
result = agent.run(task)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Harry Potter filming locations in the world, calculate the time to transfer via passenger plane to     │
│ here                                                                                                            │
│ Also give me some cricket stadiums if there are any nearby to the filming locations                             │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  locations = web_search(query="Harry Potter filming locations worldwide")                                         
  print(locations)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results
0. [Harry Potter filming locations guide - London - VisitBritain 
Shop](https://www.visitbritainshop.com/us/en/harry-potter-filming-locations-guide)
Source: visitbritainshop.com

The Renaissance Hotel has been seen worldwide by millions as a filming location for not only Harry Potter films, 
but also 102 Dalmatians, Batman, Richard III ...

1. [Top Sights Where 'Harry Potter' Was 
Filmed](https://www.ricksteves.com/watch-read-listen/read/articles/harry-potter-sites)
Source: Rick Steves Europe

Many of the movies' exterior shots — especially scenes of the Hogwarts grounds — were filmed in the craggy, cloudy 
Highlands of Scotland (mostly in the Fort ...

2. [A Magical Guide to the Best Harry Potter Places to 
Visit](https://heleneinbetween.com/2025/03/harry-potter-filming-locations-best-places-to-travel-for-potterheads.htm
l)
Source: Helene in Between

From filming locations in England and Scotland, to unique places that look just like Diagon Alley, you need to 
travel here to experience the magic.

3. [UK Harry Potter Filming Locations & 
Attractions](https://www.visitbritain.com/en/things-to-do/visit-harry-potter-filming-locations)
Source: VisitBritain

Harry Potter film locations around Britain · Freshwater West, Pembrokeshire, Wales · Lacock Abbey, Wiltshire, 
England · Alnwick Castle, Northumberland, England.

4. [20+ Magical Harry Potter Places to Visit Around the World 
...](https://whimsysoul.com/harry-potter-places-to-visit-in-real-life-filming-locations-book-inspiration/)
Date published: Jul 20, 2025
Source: Whimsy Soul

Key scenes were filmed in London (King's Cross Station, Leadenhall Market), Oxford (Christ Church, Bodleian 
Library), Scotland (Glenfinnan ...

5. [Where to find all Harry Potter Destinations Around the 
World](https://www.worldofwanderlust.com/find-harry-potter-destinations-around-world/)
Source: World of Wanderlust

One of the locations for filming Hogwarts School of Witchcraft and Wizadry was Durham Cathedral. This is also one 
of the most spectacular filming locations ...

6. ["Real World" HP Locations List : 
r/harrypotter](https://www.reddit.com/r/harrypotter/comments/78w982/real_world_hp_locations_list/)
Source: Reddit · r/harrypotter

I have compiled a list of some of the "Real World" Harry Potter locations you can visit, including filming 
locations ect. Just thought I'd share!

7. [Places in Harry Potter](https://en.wikipedia.org/wiki/Places_in_Harry_Potter)
Source: Wikipedia

Filming for subsequent films took place on a set at Leavesden Film Studios, near Watford, Hertfordshire that proved
to be cheaper than filming on location.

Out: None

[Step 1: Duration 7.04 seconds| Input tokens: 2,289 | Output tokens: 66]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Let me compile the key findings about Harry Potter filming locations and nearby cricket stadiums               
                                                                                                                   
  # Harry Potter filming locations with approximate coordinates                                                    
  hp_locations = [                                                                                                 
      {"name": "Alnwick Castle", "description": "Hogwarts Castle in Harry Potter films", "coordinates": {"lat":    
  55.4158, "lon": -1.7061}},                                                                                       
      {"name": "Durham Cathedral", "description": "Interior scenes of Hogwarts", "coordinates": {"lat": 54.7793,   
  "lon": -1.5743}},                                                                                                
      {"name": "Glenfinnan Viaduct", "description": "Hogwarts Express scenes", "coordinates": {"lat": 57.2035,     
  "lon": -5.3315}},                                                                                                
      {"name": "Leadenhall Market London", "description": "Diagon Alley", "coordinates": {"lat": 51.5128, "lon":   
  -0.0837}},                                                                                                       
      {"name": "King's Cross Station London", "description": "Platform 9¾", "coordinates": {"lat": 51.5322,        
  "lon": -0.1240}},                                                                                                
      {"name": "Christ Church Oxford", "description": "Hogwarts Great Hall", "coordinates": {"lat": 51.7589,       
  "lon": -1.2571}}                                                                                                 
  ]                                                                                                                
                                                                                                                   
  print("Harry Potter Filming Locations:")                                                                         
  for loc in hp_locations:                                                                                         
      print(f"- {loc['name']}: {loc['description']}")                                                              
                                                                                                                   
  # Key cricket stadiums near these locations                                                                      
  print("\nNearby Cricket Stadiums:")                                                                              
  print("- Durham University Ground (Chester-le-Street): Near Durham Cathedral")                                   
  [38;2;248;248;24

Execution logs:
Harry Potter Filming Locations:
- Alnwick Castle: Hogwarts Castle in Harry Potter films
- Durham Cathedral: Interior scenes of Hogwarts
- Glenfinnan Viaduct: Hogwarts Express scenes
- Leadenhall Market London: Diagon Alley
- King's Cross Station London: Platform 9¾
- Christ Church Oxford: Hogwarts Great Hall

Nearby Cricket Stadiums:
- Durham University Ground (Chester-le-Street): Near Durham Cathedral
- Lord's Cricket Ground (London): In St John's Wood
- The Oval (London): In Kennington
- Christ Church Ground (Oxford): At Christ Church College
- Alnwick Cricket Club Ground: Near Alnwick Castle
- Riverside Ground (Chester-le-Street): Near Durham Cathedral

Final answer: Harry Potter filming locations include Alnwick Castle (Hogwarts), Durham Cathedral (interior scenes),
Glenfinnan Viaduct (Hogwarts Express), Leadenhall Market (Diagon Alley), King's Cross Station (Platform 9¾), and 
Christ Church Oxford (Great Hall). Nearby cricket stadiums include Durham University Ground, Lord's Cricket Ground,
The Oval, Christ Church Ground, Alnwick Cricket Club Ground, and Riverside Ground.

[Step 5: Duration 10.16 seconds| Input tokens: 34,914 | Output tokens: 1,462]

For making your model more better at giving result

In [ ]:
agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

{task}
""")

print(detailed_report)

✌️ Splitting the task between two agents

Multi-agent structures allow to separate memories between different sub-tasks, with two great benefits:

- Each agent is more focused on its core task, thus more performant
- Separating memories reduces the count of input tokens at each step, thus reducing latency and cost.

Let's create a team with a dedicated web search agent, managed by another agent.

The manager agent should have plotting capabilities to redact its final report: so let us give it access to additional imports, including plotly, and geopandas + shapely for spatial plotting.

In [9]:
os.environ["SERPER_API_KEY"] = "20f45af533d5fb512e40a60e7e863b288a3c7253"

model = InferenceClientModel(
    "Qwen/Qwen2.5-Coder-32B-Instruct",
    provider="together",
    max_tokens=8096
)

web_agent = CodeAgent(
    model=model,
    tools=[
        GoogleSearchTool(provider="serper"),
        VisitWebpageTool(),
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=10,
)

The manager agent will need to do some mental heavy lifting.

So we give it the stronger model DeepSeek-R1, and add a planning_interval to the mix.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"

In [11]:
from smolagents import CodeAgent, InferenceClientModel, OpenAIServerModel
from smolagents.utils import encode_image_base64, make_image_url
from PIL import Image

def check_reasoning_and_plot(final_answer, agent_memory):
    multimodal_model = OpenAIServerModel("gpt-4o", max_tokens=8096)
    filepath = "saved_map.png"
    assert os.path.exists(filepath), "Make sure to save the plot under saved_map.png!"
    image = Image.open(filepath)
    prompt = (
        f"Here is a user-given task and the agent steps: {agent_memory.get_succinct_steps()}. Now here is the plot that was made."
        "Please check that the reasoning process and plot are correct: do they correctly answer the given task?"
        "First list reasons why yes/no, then write your final decision: PASS in caps lock if it is satisfactory, FAIL if it is not."
        "Don't be harsh: if the plot mostly solves the task, it should pass."
        "To pass, a plot should be made using px.scatter_map and not any other method (scatter_map looks nicer)."
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
                {
                    "type": "image_url",
                    "image_url": {"url": make_image_url(encode_image_base64(image))},
                },
            ],
        }
    ]
    output = multimodal_model(messages).content
    print("Feedback: ", output)

    if "FAIL" in output:
        raise Exception(output)

    return True



manager_agent = CodeAgent(
model=OpenAIServerModel("gpt-4o", max_tokens=8096),
tools=[calculate_cargo_travel_time],
managed_agents=[web_agent],
additional_authorized_imports=[
    "geopandas",
    "plotly",
    "plotly.io",       # explicitly add this
    "plotly.express",  # explicitly add this too
    "shapely",
    "json",
    "pandas",
    "numpy",
    "kaleido",
],
planning_interval=5,
verbosity_level=2,
final_answer_checks=[check_reasoning_and_plot],
max_steps=15,
)

In [12]:
manager_agent.visualize()

CodeAgent | gpt-4o
├── ✅ Authorized imports: ['geopandas', 'plotly', 'plotly.io', 'plotly.express', 'shapely', 'json', 'pandas', 
│   'numpy', 'kaleido']
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name                        ┃ Description                           ┃ Arguments                             ┃
│   ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ calculate_cargo_travel_time │ Calculate the travel time for a cargo │ origin_coords (`array`): Tuple of     │
│   │                             │ plane between two points on Earth     │ (latitude, longitude) for the         │
│   │                             │ using great-circle distance.          │ starting point                        │
│   │                             │                                       │ destination_coords (`array`): Tuple   │
│   │                             │                                       │ of (latitude, longitude) for the      │
│   │                             │                                       │ destination                           │
│   │                             │                                       │ cruising_speed_kmh (`number`):        │
│   │                             │                                       │ Optional cruising speed in km/h       │
│   │                             │                                       │ (defaults to 750 km/h for typical     │
│   │                             │                                       │ cargo planes)                         │
│   │ final_answer                │ Provides a final answer to the given  │ answer (`any`): The final answer to   │
│   │                             │ problem.                              │ the problem                           │
│   └─────────────────────────────┴───────────────────────────────────────┴───────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | Qwen/Qwen2.5-Coder-32B-Instruct
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Browses the web to find information
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name                        ┃ Description                       ┃ Arguments                         ┃
            ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search                  │ Performs a google web search for  │ query (`string`): The search      │
            │                             │ your query then returns a string  │ query to perform.                 │
            │                             │ of the top search results.        │ filter_year (`integer`):          │
            │                             │                                   │ Optionally restrict results to a  │
            │                             │                                   │ certain year                      │
            │ visit_webpage               │ Visits a webpage at the given url │ url (`string`): The url of the    │
            │                             │ and reads its content as a        │ webpage to visit.                 │
            │                             │ markdown string. Use this to      │                                   │
            │                             │ browse webpages.                  │                                   │
            │ calculate_cargo_travel_time │ Calculate the travel time for a   │ origin_coords (`array`): Tuple of │
            │                             │ cargo plane between two points on │ (latitude, longitude) for the     │
            │                             │ Earth                             │ starting point                    │
            │               

In [13]:
!pip install markdownify requests
!pip install -U kaleido

In [15]:
manager_agent.run("""
Find all Harry Potter filming locations in the world, calculate the time to transfer via passenger plane to here (we're in lahore 31.558° N , 74.3507° E) and return them to panda dataframe
Also give me some cricket stadiums if there are any nearby to the filming locations. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time and save it to saved_img.png

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(
    df,
    lat="centroid_lat",
    lon="centroid_lon",
    text="name",
    color="peak_hour",
    size=100,
    color_continuous_scale=px.colors.sequential.Magma,
    size_max=15,
    zoom=1
)
fig.show()
fig.write_image("saved_image.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Harry Potter filming locations in the world, calculate the time to transfer via passenger plane to     │
│ here (we're in lahore 31.558° N , 74.3507° E) and return them to panda dataframe                                │
│ Also give me some cricket stadiums if there are any nearby to the filming locations. You need at least 6 points │
│ in total.                                                                                                       │
│ Represent this as spatial map of the world, with the locations represented as scatter points with a color that  │
│ depends on the travel time and save it to saved_img.png                                                         │
│                                                                                                                 │
│ Here's an example of how to plot and return a map:                                                              │
│ import plotly.express as px                                                                                     │
│ df = px.data.carshare()                                                                                         │
│ fig = px.scatter_map(                                                                                           │
│     df,                                                                                                         │
│     lat="centroid_lat",                                                                                         │
│     lon="centroid_lon",                                                                                         │
│     text="name",                                                                                                │
│     color="peak_hour",                                                                                          │
│     size=100,                                                                                                   │
│     color_continuous_scale=px.colors.sequential.Magma,                                                          │
│     size_max=15,                                                                                                │
│     zoom=1                                                                                                      │
│ )                                                                                                               │
│ fig.show()                                                                                                      │
│ fig.write_image("saved_image.png")                                                                              │
│ final_answer(fig)                                                                                               │
│                                                                                                                 │
│ Never try to process strings using code: when you have a string to read, just print it and you'll see it.       │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o ──────────────────────────────────────────────────────────────────────────────────────────╯

KeyboardInterrupt: 